# Group 11 – Milestone 2 / Member 4 – CPU-only Benchmark

**Mục đích:** chạy mã trên branch Role 4, benchmark pipeline đầy đủ trong môi trường Google Colab CPU-only, kiểm tra ba bảng và xuất log. **Không crawl lại arXiv.** Sử dụng corpus của Milestone 1.

**Trước khi chạy:** Colab → **Runtime → Change runtime type → Hardware accelerator = CPU**; kiểm tra lại mỗi khi Runtime được reset. Sửa hai biến `BRANCH` và `DRIVE_DATA` dưới đây. Các kết quả thực tế chưa được xác minh cho đến khi notebook được chạy với corpus thật.

**Lưu ý về benchmark:** thời gian wall time đo từ lúc khởi động `pipeline_m2.py` tới lúc hoàn thành xuất Parquet, không bao gồm việc mount Drive, copy data, chạy validation và thống kê sau đó. RAM là RSS lấy mẫu của process và con; các trang bộ nhớ chia sẻ có thể bị cộng nhiều lần.

## 1. Xác minh môi trường CPU và thông tin phiên chạy

In [ ]:
import os, sys, shutil, platform, subprocess, json, datetime
from pathlib import Path
import psutil
print("UTC time:", datetime.datetime.now(datetime.timezone.utc).isoformat())
print("Python:", sys.version)
print("CPU logical:", psutil.cpu_count(logical=True))
print("Total RAM (GB):", round(psutil.virtual_memory().total/1e9, 2))
print("Platform:", platform.platform())
print("COLAB_GPU env:", os.environ.get("COLAB_GPU", "not set"))
print("nvidia-smi path:", shutil.which("nvidia-smi"))
print("MANUAL CHECK REQUIRED: Runtime -> Change runtime type -> CPU")

## 2. Mount Google Drive – nơi chứa dữ liệu M1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone đúng branch Role 4 đã push

In [ ]:
REPO_URL = "https://github.com/mituleefam/group11-arxiv-project.git"
BRANCH = "tram-m2-member4"  # branch chứa code đã được review
REPO_DIR = Path("/content/group11-arxiv-project")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
actual_branch = subprocess.check_output(["git", "-C", str(REPO_DIR), "branch", "--show-current"], text=True).strip()
assert actual_branch == BRANCH, f"Expected {BRANCH}, found {actual_branch}; reopen with a clean runtime"
print("Branch:", actual_branch)
print("Commit:", subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "src" / "requirements.txt")], check=True)
import pyarrow
print("pyarrow:", pyarrow.__version())


## 4. Nạp corpus M1 từ thư mục Drive hoặc file `Tram_MS2.tar.gz` (chỉ xử lý bản sao trên `/content`)


In [ ]:
# SỬA: trỏ tới (1) thư mục có corpus/ hoặc (2) file nén Tram_MS2.tar.gz.
DRIVE_DATA = Path("/content/drive/MyDrive/EDIT_ME/Tram_MS2.tar.gz")
WORK_DATA = Path("/content/m2_member4_working_data")
assert DRIVE_DATA.exists(), f"Missing M1 data source: {DRIVE_DATA}"
assert not WORK_DATA.resolve().is_relative_to(DRIVE_DATA.resolve()), "Never modify original Drive data"
assert not WORK_DATA.exists(), "Working data already exists. Review it before creating a new benchmark copy."

if DRIVE_DATA.is_file():
    assert DRIVE_DATA.name.endswith((".tar.gz", ".tgz")), "Expected a .tar.gz archive"
    import tarfile
    extract_root = Path("/content/m2_member4_extracted")
    assert not extract_root.exists(), "Extracted folder already exists. Restart clean runtime to avoid mixed data."
    extract_root.mkdir(parents=True)
    with tarfile.open(DRIVE_DATA, mode="r:gz") as archive:
        # Ignore symlinks and unsafe paths; only accept files/folders from the M1 archive.
        archive.extractall(extract_root, filter="data")
    candidates = [folder.parent for folder in extract_root.rglob("corpus") if folder.is_dir()]
    assert len(candidates) == 1, f"Expected exactly one corpus folder; found {candidates}"
    source = candidates[0]
else:
    source = DRIVE_DATA

assert (source / "corpus").is_dir(), f"Input folder missing: {source / 'corpus'}"
WORK_DATA.mkdir(parents=True)
shutil.copytree(source / "corpus", WORK_DATA / "corpus", ignore=shutil.ignore_patterns("parsed"))
(WORK_DATA / "work").mkdir(exist_ok=True)
manifest = source / "work" / "paper_manifest.jsonl"
if manifest.exists():
    shutil.copy2(manifest, WORK_DATA / "work" / "paper_manifest.jsonl")
count = sum(p.is_dir() for p in (WORK_DATA / 'corpus').iterdir())
print("Input paper folders:", count)
assert count > 0, "No paper folders found"
print("Local working data:", WORK_DATA)


## 5. Chạy unit test và kiểm tra contract trước khi chạy corpus

In [ ]:
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "src",
                "-p", "test_m2_member4.py", "-v"], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, "src/test_m2_member1.py"], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, "src/reference_parser/test_m2_member3.py"], cwd=REPO_DIR, check=True)

## 6. Chạy thử 5 bài và xem module nào còn stand-in

In [ ]:
subprocess.run([sys.executable, "src/pipeline_m2.py", "--data-dir", str(WORK_DATA),
                "--limit", "5", "--workers", "2", "--force"], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, "src/validate_m2.py", "--data-dir", str(WORK_DATA),
                "--limit", "5", "--require-parquet"], cwd=REPO_DIR, check=True)
print("Check the output: do NOT accept a full-group benchmark if Member 2/3 stand-ins remain.")

## 7. Benchmark chính thức – chạy full corpus trên CPU

Cell này **chỉ xóa kết quả M2 trong bản copy `/content/m2_member4_working_data`** để đo từ đầu, không động đến `DRIVE_DATA`. Kiểm tra kỹ `WORK_DATA` trước khi chạy.

**Nếu Colab bị ngắt:** log còn lại không chứng minh run hoàn tất. Đừng gọi số liệu của run bị ngắt là wall time end-to-end đầy đủ; tiếp tục theo checkpoint nếu cần, và ghi rõ giới hạn của số liệu.

In [ ]:
# Chỉ xóa parsed/tables/logs trong WORK_DATA, không xóa corpus .tex/.bib của M1.
assert WORK_DATA == Path("/content/m2_member4_working_data")
for folder in (WORK_DATA / "corpus").iterdir():
    if folder.is_dir():
        shutil.rmtree(folder / "parsed", ignore_errors=True)
shutil.rmtree(WORK_DATA / "tables", ignore_errors=True)
shutil.rmtree(WORK_DATA / "logs", ignore_errors=True)
command = [sys.executable, "src/benchmark_m2.py", "--data-dir", str(WORK_DATA),
           "--workers", "2", "--force", "--interval", "0.5", "--disk-interval", "30",
           "--environment", "google_colab", "--cpu-only-declared"]
subprocess.run(command, cwd=REPO_DIR, check=True)

## 8. Nghiệm thu Parquet + JSON sau khi chạy full

In [ ]:
subprocess.run([sys.executable, "src/validate_m2.py", "--data-dir", str(WORK_DATA),
                "--require-parquet"], cwd=REPO_DIR, check=True)
print("Validation has no FAIL. WARN requires review, but does not automatically mean the parser failed.")

## 9. Tái lập thống kê BibTeX trước/sau dedup của Member 3

In [ ]:
subprocess.run([sys.executable, "src/reference_parser/reference_stats.py",
                "--data-dir", str(WORK_DATA)], cwd=REPO_DIR, check=True)

## 10. Tổng hợp thống kê, đối chiếu dung lượng counterfactual và xuất file CSV

In [ ]:
subprocess.run([sys.executable, "src/statistics_m2.py", "--data-dir", str(WORK_DATA),
                "--counterfactual-parquet"], cwd=REPO_DIR, check=True)
import csv
for file in ["validation_report_m2.csv", "performance_log_m2.csv", "summary_statistics_m2.csv"]:
    path = WORK_DATA / "logs" / file
    print(file, path.exists(), path.stat().st_size if path.exists() else None)

## 11. Lưu log lên Drive cho Member 5 (không commit corpus hay bảng Parquet lớn)

In [ ]:
LOG_ARCHIVE = Path("/content/m2_role4_logs.zip")
shutil.make_archive(str(LOG_ARCHIVE.with_suffix("")), "zip", root_dir=WORK_DATA, base_dir="logs")
EXPORT_DIR = Path("/content/drive/MyDrive/m2_role4_report_logs")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(LOG_ARCHIVE, EXPORT_DIR / LOG_ARCHIVE.name)
print("Logs stored:", EXPORT_DIR / LOG_ARCHIVE.name)
print("Do not push this archive or raw corpus to GitHub source repository.")